# V7.0: Boundary Loss + Hierarchy Loss

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'Data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mon Mar 30 05:52:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             58W /  400W |       0MiB /  40960MiB |      0%      Default |
|          

In [2]:
import os
os.chdir(REPO_DIR)
print('Smoke test: V7.0 (2 samples, 1 epoch)')
!python -u train.py \
    --config configs/autoresearch/V7.0_boundary_hierarchy.yaml \
    --max-samples 2 \
    --max-epochs 1 \
    --no-text-ratio 0.0 \
    --grad-accum 1

Smoke test: V7.0 (2 samples, 1 epoch)
2026-03-30 05:52:59.977901: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 05:53:00.494644: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774849980.724467   55692 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774849980.793035   55692 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774849981.226662   55692 computation_placer.cc:177] computation placer already registered. P

## V7.0 Training

Resume from V5.0, reset optimizer, 80 epochs with Boundary Loss annealing.

In [3]:
import os, glob
os.chdir(REPO_DIR)
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
V50_CKPT = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
assert os.path.exists(V50_CKPT), f'V5.0 not found: {V50_CKPT}'
for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Training V7.0...')
!python -u train.py \
    --config configs/autoresearch/V7.0_boundary_hierarchy.yaml \
    --resume "{V50_CKPT}" \
    --reset-optimizer \
    --reset-lr \
    --no-text-ratio 0.15 \
    --grad-accum 2
sync_and_tag('V7.0')
print('V7.0 training complete!')

Training V7.0...
2026-03-30 05:56:40.959875: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 05:56:40.978840: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774850201.001730   57003 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774850201.009273   57003 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774850201.028940   57003 computation_placer.cc:177] computation placer already registered. Please check linkage a

## Evaluation

In [4]:
import subprocess, re, os
os.chdir(REPO_DIR)
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
ckpt = os.path.join(DRIVE_CKPT, 'best_V7.0.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')
CONFIG = 'configs/autoresearch/V7.0_boundary_hierarchy.yaml'
BASELINE = {'ET': 0.7910, 'TC': 0.8560, 'WT': 0.8967, 'Mean': 0.8479}
for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py', '--config', CONFIG, '--checkpoint', ckpt, '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    for line in ret.stdout.split('\n'):
        if 'dice_' in line or 'hd95_' in line:
            print(f'  {line.strip()}')
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-300:]}')
print()
print(f'Baseline V5.0: ET={BASELINE["ET"]}, TC={BASELINE["TC"]}, WT={BASELINE["WT"]}, Mean={BASELINE["Mean"]}')

Using: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V7.0.pth

text+TTA
  dice_ET: 0.7848 +/- 0.2047
  dice_TC: 0.8570 +/- 0.1587
  dice_WT: 0.9032 +/- 0.0563
  dice_mean: 0.8483 +/- 0.0990
  hd95_ET: 3.02 +/- 7.16
  hd95_TC: 2.41 +/- 3.96
  hd95_WT: 1.92 +/- 4.87

notext+TTA
  dice_ET: 0.7825 +/- 0.2029
  dice_TC: 0.8475 +/- 0.1696
  dice_WT: 0.8978 +/- 0.0592
  dice_mean: 0.8426 +/- 0.1025
  hd95_ET: 3.10 +/- 7.13
  hd95_TC: 2.52 +/- 3.98
  hd95_WT: 2.02 +/- 5.10

Baseline V5.0: ET=0.791, TC=0.856, WT=0.8967, Mean=0.8479
